In [ ]:
# DONE
# ABOUT 6 HOURS

# 2. LOCAL EFFECTIVE DIMENSION FOR QNN AND EASY QNN
# calculated around the space parameter found after training the QNN/easy QNN

# Trained weights: run a quick VQC fit to get trained parameters
def get_trained_weights(feature_map_fn, ansatz_fn, N, x_tr, y_tr, maxiter=100):
    trained_weights_store = []
    def adam_capture(fun, x0, jac=None, **kwargs):
        opt = ADAM(maxiter=maxiter, lr=0.1, beta_1=0.9, beta_2=0.99, tol=1e-8)
        def tracked_jac(w):
            trained_weights_store.append(w.copy())
            return jac(w)
        return opt.minimize(fun=fun, x0=x0, jac=tracked_jac)
    vqc = VQC(
        sampler=StatevectorSampler(),
        feature_map=feature_map_fn(N),
        ansatz=ansatz_fn(N),
        optimizer=adam_capture,
    )
    vqc.fit(x_tr, y_tr)
    return trained_weights_store[-1] if trained_weights_store else vqc.weights

# Training QNNs for local ED
print('Training QNN for local ED...')
trained_weights_qnn  = get_trained_weights(FeatureMap, Ansatz_60_par, N, X_s_tr, y_s_tr)
print('Training easy QNN for local ED...')
trained_weights_easy = get_trained_weights(EasyFeatureMap, Ansatz_60_par, N, X_s_tr, y_s_tr)

# Local ED — QNN
local_ed_trained_qnn   = LocalEffectiveDimension(qnn=qnn_s,      weight_samples=trained_weights_qnn,  input_samples=X_s_tr)

# Local ED — easy QNN
local_ed_trained_easy   = LocalEffectiveDimension(qnn=qnn_s_easy, weight_samples=trained_weights_easy, input_samples=X_s_tr)

local_eff_trained_qnn    = local_ed_trained_qnn.get_effective_dimension(dataset_size=n)

local_eff_trained_easy   = local_ed_trained_easy.get_effective_dimension(dataset_size=n)


# save data
with open("object_local_ed_trained_qnn_syn.pkl", "wb") as f:
    pickle.dump(local_ed_trained_qnn, f)

with open("object_local_ed_trained_easy_syn.pkl", "wb") as f:
    pickle.dump(local_ed_trained_easy, f)

with open("local_eff_trained_qnn_syn.pkl", "wb") as f:
    pickle.dump(local_eff_trained_qnn, f)
with open("local_eff_trained_easy_syn.pkl", "wb") as f:
    pickle.dump(local_eff_trained_easy, f)

In [ ]:
# FISHER INFORMATION FOR QNN AND EASY QNN

# load data
with open("object_local_ed_trained_qnn_syn.pkl", "rb") as f:
    local_eff_trained_qnn = pickle.load(f)

with open("object_local_ed_trained_easy_syn.pkl", "rb") as f:
    local_eff_trained_easy = pickle.load(f)

# compute Fisher information
def compute_fim(ed_obj):
    """Run MC once, return FIM averaged over input samples. Shape: (n_weights, n_weights)"""
    grads, outputs = ed_obj.run_monte_carlo()
    fim = ed_obj.get_fisher_information(grads, outputs)  # (n_samples, n_w, n_w)
    
    return fim.mean(axis=0)                               # (n_w, n_w)

fim_trained_qnn    = compute_fim(local_eff_trained_qnn)

fim_trained_easy   = compute_fim(local_eff_trained_easy)


print(f'FIM trace QNN   trained: {np.trace(fim_trained_qnn):.4f}')

print(f'FIM trace easy   trained: {np.trace(fim_trained_easy):.4f}')


# save Fisher Information
with open("fim_trained_qnn_syn.pkl", "wb") as f:
    pickle.dump(fim_trained_qnn, f)
with open("fim_trained_easy_syn.pkl", "wb") as f:
    pickle.dump(fim_trained_easy, f)